In [4]:
from __future__ import annotations

import json
import os
import random
import re
import time
import uuid
from dataclasses import asdict, dataclass, field
from enum import Enum
from pathlib import Path
from typing import Any, Callable, Optional

import numpy as np
import pandas as pd
os.environ["USE_HF"] = "1"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"项目目录: {PROJECT_ROOT.resolve()}")
print("USE_HF:", os.getenv("USE_HF", "0"), "（默认 0，离线可运行）")

项目目录: D:\CodeData\Program Coding\Project\Writing_Coach_Agent
USE_HF: 1 （默认 0，离线可运行）


In [2]:
class StepStatus(str, Enum):
    PENDING = "pending"
    RUNNING = "running"
    SUCCEEDED = "succeeded"
    FAILED = "failed"


@dataclass
class AgentState:
    task: str
    inputs: dict[str, Any]
    run_id: str = field(default_factory=lambda: uuid.uuid4().hex[:8])
    artifacts: dict[str, Any] = field(default_factory=dict)
    trace: list[dict[str, Any]] = field(default_factory=list)
    final_answer: Optional[dict[str, Any]] = None

    def log(self, event: str, **payload: Any) -> None:
        self.trace.append({
            "time": time.strftime("%H:%M:%S"),
            "run_id": self.run_id,
            "event": event,
            **payload,
        })


@dataclass(frozen=True)
class ToolSpec:
    name: str
    purpose: str
    required_inputs: tuple[str, ...]
    output_fields: tuple[str, ...]
    errors: tuple[str, ...]
    version: str
    source: str


class ToolRegistry:
    def __init__(self) -> None:
        self._tools: dict[str, Callable[..., Any]] = {}
        self._specs: dict[str, ToolSpec] = {}

    def register(self, spec: ToolSpec, func: Callable[..., Any]) -> None:
        if spec.name in self._tools:
            raise ValueError(f"工具已注册: {spec.name}")
        self._tools[spec.name] = func
        self._specs[spec.name] = spec

    def call(self, name: str, **kwargs: Any) -> Any:
        if name not in self._tools:
            raise KeyError(f"未知工具: {name}")
        missing = [x for x in self._specs[name].required_inputs if x not in kwargs]
        if missing:
            raise ValueError(f"{name} 缺少必填输入: {missing}")
        result = self._tools[name](**kwargs)
        json.dumps(result, ensure_ascii=False)  # 工具边界：结果必须可 JSON 序列化
        return result

    @property
    def specs(self) -> list[ToolSpec]:
        return [self._specs[name] for name in sorted(self._specs)]

In [6]:
def read_jsonl(path: Path) -> list[dict[str, Any]]:
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


essays = read_jsonl(DATA_DIR / "essays.jsonl")
keystroke_rows = read_jsonl(DATA_DIR / "keystrokes.jsonl")
keystrokes = {row["essay_id"]: row for row in keystroke_rows}
rubric = json.loads((DATA_DIR / "rubric.jsonl").read_text(encoding="utf-8"))

essay_ids = {row["essay_id"] for row in essays}
log_ids = set(keystrokes)
audit = pd.DataFrame([
    {"check": "essay_count", "value": len(essays), "passed": len(essays) > 0},
    {"check": "keystroke_count", "value": len(keystrokes), "passed": len(keystrokes) > 0},
    {"check": "id_alignment", "value": sorted(essay_ids ^ log_ids), "passed": essay_ids == log_ids},
    {"check": "rubric_dimensions", "value": sorted(rubric), "passed": {"language", "argumentation"} <= set(rubric)},
])
display(audit)
assert audit["passed"].all(), "数据审计未通过，请先修复主键或 Rubric。"

,check,value,passed
0,essay_count,4,True
1,keystroke_count,4,True
2,id_alignment,[],True
3,rubric_dimensions,"[argumentation, language]",True


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


class SimilarityBackend:
    def __init__(self) -> None:
        self.mode = "tfidf"
        self.model = None
        if os.getenv("USE_HF", "0") == "1":
            try:
                from sentence_transformers import SentenceTransformer
                self.model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
                self.mode = "huggingface"
            except Exception as exc:
                print(f"Hugging Face 模型不可用，自动降级为 TF-IDF：{exc}")

    def rank(self, query: str, candidates: list[str]) -> list[tuple[int, float]]:
        if not candidates:
            return []
        if self.mode == "huggingface":
            vectors = self.model.encode([query] + candidates, normalize_embeddings=True)
            scores = vectors[1:] @ vectors[0]
        else:
            matrix = TfidfVectorizer(ngram_range=(1, 2), stop_words="english").fit_transform(
                [query] + candidates
            )
            scores = cosine_similarity(matrix[0:1], matrix[1:]).ravel()
        return sorted(enumerate(scores), key=lambda item: item[1], reverse=True)


similarity = SimilarityBackend()
print("相似度后端:", similarity.mode)

d:\CodeData\software\Miniconda3\envs\writing_coach_agent\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\CodeData\software\Miniconda3\envs\writing_coach_agent\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\15207\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or t

相似度后端: huggingface


In [8]:
def sentence_spans(text: str) -> list[dict[str, Any]]:
    spans = []
    for index, match in enumerate(re.finditer(r"[^.!?]+[.!?]?", text)):
        sentence = match.group(0).strip()
        if not sentence:
            continue
        start = text.find(sentence, match.start(), match.end())
        spans.append({
            "sentence_index": index,
            "sentence": sentence,
            "char_start": start,
            "char_end": start + len(sentence),
        })
    return spans

In [9]:
def text_analysis_tool(essay: str) -> dict[str, Any]:
    if not isinstance(essay, str) or not essay.strip():
        raise ValueError("essay 必须是非空字符串")
    words = re.findall(r"\b[A-Za-z']+\b", essay)
    sentences = sentence_spans(essay)
    transition_terms = ["first", "second", "however", "therefore", "because", "for example", "in conclusion"]
    transitions = [term for term in transition_terms if re.search(rf"\b{re.escape(term)}\b", essay, re.I)]
    counts: dict[str, int] = {}
    for word in words:
        counts[word.lower()] = counts.get(word.lower(), 0) + 1
    repeated = sorted(word for word, count in counts.items() if count >= 4)
    return {
        "word_count": len(words),
        "sentence_count": len(sentences),
        "avg_sentence_length": round(len(words) / max(1, len(sentences)), 2),
        "transition_terms": transitions,
        "over_repeated_words": repeated[:8],
        "has_counterargument": bool(re.search(r"\b(however|although|some people|on the other hand)\b", essay, re.I)),
        "has_example": bool(re.search(r"\b(for example|for instance|such as)\b", essay, re.I)),
        "has_conclusion": bool(re.search(r"\b(therefore|in conclusion|to conclude)\b", essay, re.I)),
    }

In [10]:
def process_analysis_tool(log: dict[str, Any]) -> dict[str, Any]:
    required = {"total_time_sec", "preparation_time_sec", "insertions", "deletions", "replacements"}
    missing = sorted(required - set(log))
    if missing:
        raise ValueError(f"Keystroke log 缺少字段: {missing}")
    raw_insertions = int(log["insertions"])
    insertions = max(raw_insertions, 1)
    total_time = max(float(log["total_time_sec"]), 1.0)
    pauses = np.asarray(log.get("pauses_ms", []), dtype=float)
    return {
        "preparation_ratio": round(float(log["preparation_time_sec"]) / total_time, 3),
        "deletion_insertion_ratio": round(float(log["deletions"]) / insertions, 3),
        "revision_operations": int(log["deletions"] + log["replacements"]),
        "long_pause_count": int((pauses >= 2000).sum()),
        "median_pause_ms": round(float(np.median(pauses)), 1) if len(pauses) else 0.0,
        "writing_efficiency": round(raw_insertions / total_time, 3),
        "data_quality": "warning_zero_insertions" if raw_insertions == 0 else "ok",
    }

In [17]:
def rubric_retrieval_tool(dimension: str, rubric: dict[str, Any]) -> dict[str, Any]:
    if dimension not in rubric:
        raise ValueError(f"未知评分维度: {dimension}；可用维度: {sorted(rubric)}")
    return {
        "dimension": dimension,
        "levels": rubric[dimension],
        "version": "teaching-rubric-v1",
        "source": "data/rubric.jsonl",
    }

In [12]:
comparison_rows = []
for essay_id in ["E001", "E004"]:
    record = next(row for row in essays if row["essay_id"] == essay_id)
    text_features = text_analysis_tool(record["essay"])
    process_features = process_analysis_tool(keystrokes[essay_id])
    comparison_rows.append({
        "essay_id": essay_id,
        "word_count": text_features["word_count"],
        "has_example": text_features["has_example"],
        "has_conclusion": text_features["has_conclusion"],
        "revision_operations": process_features["revision_operations"],
        "preparation_ratio": process_features["preparation_ratio"],
    })

display(pd.DataFrame(comparison_rows))

,essay_id,word_count,has_example,has_conclusion,revision_operations,preparation_ratio
0,E001,57,True,True,59,0.083
1,E004,25,False,False,2,0.027


In [13]:
def evidence_locator_tool(essay: str, focus: str, top_k: int = 2) -> list[dict[str, Any]]:
    if top_k < 1:
        raise ValueError("top_k 必须大于等于 1")
    spans = sentence_spans(essay)
    ranked = similarity.rank(focus, [item["sentence"] for item in spans])[:top_k]
    evidence = []
    for idx, score in ranked:
        item = dict(spans[idx])
        item.update({"relevance": round(float(score), 4), "backend": similarity.mode})
        evidence.append(item)
    return evidence


evidence_sample = next(row for row in essays if row["essay_id"] == "E003")
query_results = {}
for focus in [
    "main claim reason evidence conclusion",
    "counterargument limitation alternative",
]:
    query_results[focus] = evidence_locator_tool(evidence_sample["essay"], focus, top_k=2)

for focus, results in query_results.items():
    print("\nFOCUS:", focus)
    display(pd.DataFrame(results)[["sentence_index", "sentence", "relevance", "char_start", "char_end", "backend"]])
    assert all(evidence_sample["essay"][x["char_start"]:x["char_end"]] == x["sentence"] for x in results)


FOCUS: main claim reason evidence conclusion


,sentence_index,sentence,relevance,char_start,char_end,backend
0,2,Schools could instead recognize progress throu...,0.2322,169,277,huggingface
1,0,Cash rewards may motivate students temporarily...,0.1341,0,94,huggingface



FOCUS: counterargument limitation alternative


,sentence_index,sentence,relevance,char_start,char_end,backend
0,3,These alternatives reward effort without turni...,0.2219,278,355,huggingface
1,1,A student who studies only for money may stop ...,0.1572,95,168,huggingface


In [14]:
def scoring_tool(text_features: dict[str, Any], process_features: dict[str, Any]) -> dict[str, Any]:
    language = 1.0
    language += min(text_features["word_count"] / 50, 1.3)
    language += min(text_features["avg_sentence_length"] / 18, 0.8)
    language += min(len(text_features["transition_terms"]) * 0.2, 0.8)
    language -= min(len(text_features["over_repeated_words"]) * 0.15, 0.5)

    argumentation = 1.0
    argumentation += 0.9 if text_features["has_example"] else 0
    argumentation += 0.9 if text_features["has_counterargument"] else 0
    argumentation += 0.7 if text_features["has_conclusion"] else 0
    argumentation += min(len(text_features["transition_terms"]) * 0.18, 0.7)
    argumentation += 0.4 if process_features["revision_operations"] >= 15 else 0
    return {
        "language": round(float(np.clip(language, 1, 5)), 2),
        "argumentation": round(float(np.clip(argumentation, 1, 5)), 2),
        "score_type": "可解释启发式演示分数，不代表训练模型预测",
    }


specs = [
    ToolSpec("text_analysis", "提取作文文本特征", ("essay",), ("word_count", "sentence_count"), ("ValueError",), "1.1", "essay_text"),
    ToolSpec("process_analysis", "提取写作过程特征", ("log",), ("preparation_ratio", "revision_operations"), ("ValueError",), "1.1", "keystroke_log"),
    ToolSpec("rubric_retrieval", "按维度读取评分标准", ("dimension", "rubric"), ("dimension", "levels", "version"), ("ValueError",), "1.1", "rubric_kb"),
    ToolSpec("evidence_locator", "定位原文证据", ("essay", "focus"), ("sentence", "char_start", "char_end"), ("ValueError",), "1.1", "essay_text"),
    ToolSpec("scoring", "生成演示分数", ("text_features", "process_features"), ("language", "argumentation"), tuple(), "1.0", "derived_features"),
]
functions = [text_analysis_tool, process_analysis_tool, rubric_retrieval_tool, evidence_locator_tool, scoring_tool]

registry = ToolRegistry()
for spec, func in zip(specs, functions):
    registry.register(spec, func)

display(pd.DataFrame(asdict(spec) for spec in registry.specs)[["name", "required_inputs", "output_fields", "version", "source"]])

,name,required_inputs,output_fields,version,source
0,evidence_locator,"(essay, focus)","(sentence, char_start, char_end)",1.1,essay_text
1,process_analysis,"(log,)","(preparation_ratio, revision_operations)",1.1,keystroke_log
2,rubric_retrieval,"(dimension, rubric)","(dimension, levels, version)",1.1,rubric_kb
3,scoring,"(text_features, process_features)","(language, argumentation)",1.0,derived_features
4,text_analysis,"(essay,)","(word_count, sentence_count)",1.1,essay_text


In [18]:
def run_diagnostic_agent(essay_record: dict[str, Any], process_log: dict[str, Any]) -> AgentState:
    state = AgentState(
        task="生成带 Rubric 与 Evidence 的多源作文诊断",
        inputs={"essay": essay_record["essay"], "prompt": essay_record["prompt"], "log": process_log},
    )
    state.log("run_started", essay_id=essay_record["essay_id"])

    text_features = registry.call("text_analysis", essay=essay_record["essay"])
    state.log("tool_succeeded", tool="text_analysis")
    process_features = registry.call("process_analysis", log=process_log)
    state.log("tool_succeeded", tool="process_analysis")
    rubrics = {
        dimension: registry.call("rubric_retrieval", dimension=dimension, rubric=rubric)
        for dimension in ["language", "argumentation"]
    }
    state.log("tool_succeeded", tool="rubric_retrieval", dimensions=list(rubrics))

    evidence = {
        "claim_and_reasons": registry.call(
            "evidence_locator", essay=essay_record["essay"],
            focus="main claim reason evidence argument conclusion", top_k=3,
        ),
        "language_control": registry.call(
            "evidence_locator", essay=essay_record["essay"],
            focus="varied clear precise sentence transition", top_k=2,
        ),
    }
    state.log("tool_succeeded", tool="evidence_locator", backend=similarity.mode)
    scores = registry.call("scoring", text_features=text_features, process_features=process_features)
    state.log("tool_succeeded", tool="scoring")
    suggestions = []
    claim_ids = [item["sentence_index"] for item in evidence["claim_and_reasons"]]
    if not text_features["has_example"]:
        suggestions.append({
            "dimension": "argumentation",
            "action": "为核心理由补充一个具体例子或可验证事实。",
            "reason_feature": "has_example=false",
            "evidence_ids": claim_ids,
        })
    if not text_features["has_counterargument"]:
        suggestions.append({
            "dimension": "argumentation",
            "action": "加入一个反方观点，并解释其局限。",
            "reason_feature": "has_counterargument=false",
            "evidence_ids": claim_ids,
        })
    if process_features["revision_operations"] < 15:
        suggestions.append({
            "dimension": "process",
            "action": "完成初稿后安排一次结构性修订，而不只修改拼写。",
            "reason_feature": "revision_operations<15",
            "evidence_ids": [],
        })
    if text_features["word_count"] < 80:
        suggestions.append({
            "dimension": "argumentation",
            "action": "扩展理由链：主张 → 原因 → 证据 → 解释。",
            "reason_feature": "word_count<80",
            "evidence_ids": claim_ids,
        })

    report = {
        "essay_id": essay_record["essay_id"],
        "scores": scores,
        "text_features": text_features,
        "process_features": process_features,
        "rubric": rubrics,
        "evidence": evidence,
        "suggestions": suggestions,
        "provenance": {
            "essay_source": "data/essays.jsonl",
            "process_source": "data/keystrokes.jsonl",
            "rubric_source": "data/rubric.jsonl",
            "evidence_backend": similarity.mode,
        },
    }
    state.artifacts.update(report)
    state.final_answer = report
    state.log("run_finished", success=True, suggestion_count=len(suggestions))
    return state

In [19]:
sample = next(row for row in essays if row["essay_id"] == "E002")
state = run_diagnostic_agent(sample, keystrokes[sample["essay_id"]])
print(json.dumps(state.final_answer, ensure_ascii=False, indent=2))

trace_df = pd.DataFrame(state.trace)
display(trace_df[[column for column in ["time", "event", "tool", "backend", "success", "suggestion_count"] if column in trace_df.columns]])

{
  "essay_id": "E002",
  "scores": {
    "language": 2.61,
    "argumentation": 2.96,
    "score_type": "可解释启发式演示分数，不代表训练模型预测"
  },
  "text_features": {
    "word_count": 39,
    "sentence_count": 5,
    "avg_sentence_length": 7.8,
    "transition_terms": [
      "because",
      "in conclusion"
    ],
    "over_repeated_words": [],
    "has_counterargument": true,
    "has_example": false,
    "has_conclusion": true
  },
  "process_features": {
    "preparation_ratio": 0.039,
    "deletion_insertion_ratio": 0.042,
    "revision_operations": 9,
    "long_pause_count": 0,
    "median_pause_ms": 450.0,
    "writing_efficiency": 0.613,
    "data_quality": "ok"
  },
  "rubric": {
    "language": {
      "dimension": "language",
      "levels": {
        "1": "Frequent grammatical errors and limited vocabulary make meaning difficult to follow.",
        "3": "Generally clear language with some errors and adequate sentence variety.",
        "5": "Precise vocabulary, varied sentence structu

,time,event,tool,backend,success,suggestion_count
0,10:10:35,run_started,NaN,NaN,NaN,NaN
1,10:10:35,tool_succeeded,text_analysis,NaN,NaN,NaN
2,10:10:35,tool_succeeded,process_analysis,NaN,NaN,NaN
3,10:10:35,tool_succeeded,rubric_retrieval,NaN,NaN,NaN
4,10:10:35,tool_succeeded,evidence_locator,huggingface,NaN,NaN
5,10:10:35,tool_succeeded,scoring,NaN,NaN,NaN
6,10:10:35,run_finished,NaN,NaN,True,3.0


In [20]:
# 1) 空作文：明确输入错误
try:
    text_analysis_tool("   ")
    raise AssertionError("空作文测试应失败")
except ValueError as exc:
    assert "非空字符串" in str(exc)

# 2) 零插入与空停顿：无除零错误，并保留数据质量警告
edge_log = {
    "total_time_sec": 0,
    "preparation_time_sec": 0,
    "insertions": 0,
    "deletions": 0,
    "replacements": 0,
    "pauses_ms": [],
}
edge_features = process_analysis_tool(edge_log)
assert edge_features["deletion_insertion_ratio"] == 0
assert edge_features["median_pause_ms"] == 0
assert edge_features["data_quality"] == "warning_zero_insertions"

# 3) 非法 Rubric 维度：给出可解释异常
try:
    rubric_retrieval_tool("fluency", rubric)
    raise AssertionError("非法维度测试应失败")
except ValueError as exc:
    assert "可用维度" in str(exc)

# 4) top_k 大于句子数：返回全部可用句子，不越界
short_text = "One claim. One reason."
assert len(evidence_locator_tool(short_text, "claim reason", top_k=10)) == 2

print("✅ 边界测试通过：空输入、零插入、空停顿、非法维度与超大 top_k 均有稳定行为。")

✅ 边界测试通过：空输入、零插入、空停顿、非法维度与超大 top_k 均有稳定行为。
